## Milestone 3

In [49]:
#Milestone 3 setup


!pip install faiss-cpu #install FAISS

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('train.csv')

print("Creating nowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")


Creating nowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base successfully created


### Zero-shot classifier for Q1, Q2, Q6

In [50]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [51]:
result = zs(prompt_150, candidate_labels=labels_150)
score = result['scores'][result['labels'].index(ans_150)]
score

0.38441911339759827

Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [52]:
query = model.encode([prompt_150])
D, I = index.search(np.array(query), 10)

retrieved_indices = I[0]
rank = list(retrieved_indices).index(150)+1
rank

10

Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder.
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context.

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

Cross Encoders (Hugging Face)


In [53]:
#Code to use a Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [54]:
order = np.argsort(ce_scores)[::-1]
reranked = retrieved_indices[order]

rank = list(reranked).index(150) + 1
rank

1

Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?


In [55]:
row = train.iloc[42]
prompt = str(row["prompt"])

query = model.encode([prompt])
D, I = index.search(np.array(query), 5)
docs = [kb[i] for i in I[0]]

context = " ".join(docs)
text = "Context: " + context + " Question: " + prompt

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tokenizer(text, truncation=False)
len(tokens["input_ids"])

216

Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).


In [56]:
true_doc = kb[150]
rag = "Context: " + true_doc + " Question: " + prompt_150

result = zs(rag, candidate_labels=labels_150)
score = result["scores"][result["labels"].index(ans_150)]
score

0.9894258379936218

Concept: The Danger of Bad Retrieval (Adversarial RAG)
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.


Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).


In [57]:
wrong_doc = kb[999]
rag = "Context: " + wrong_doc + " Question: " + prompt_150

result = zs(rag, candidate_labels=labels_150)
score = result["scores"][result["labels"].index(ans_150)]
score

0.5289478302001953

In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).


In [60]:
hits = 0

for i in range(100):
  row = train.iloc[i]
  prompt = str(row["prompt"])
  answer = str(row[row["answer"]])

  query = model.encode([prompt])
  D, I = index.search(np.array(query), 5)
  docs = [kb[x] for x in I[0]]

  found = False

  for doc in docs:
    if answer in doc:
      found = True
      break

  if found:
    hits += 1

hits

73

Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [61]:
total_map = 0

for i in range(20):
  row = train.iloc[i]
  prompt = str(row["prompt"])

  labels = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]
  correct = row["answer"]
  answer = str(row[correct])

  query = model.encode([prompt])
  D, I = index.search(np.array(query), 5)
  docs = [kb[x] for x in I[0]]
  pairs = []

  for doc in docs:
    pairs.append([prompt, doc])

  scores = cross_encoder.predict(pairs)
  best_doc = docs[np.argmax(scores)]
  rag = "Context: " + best_doc + " Question: " + prompt

  result = zs(rag, candidate_labels=labels)
  ranked_labels = []

  for label in result["labels"]:
    ranked_labels.append(chr(ord("A") + labels.index(label)))

  top3 = ranked_labels[:3]

  if correct == top3[0]:
    ap = 1
  elif correct == top3[1]:
    ap = 1 / 2
  elif correct == top3[2]:
    ap = 1 / 3
  else:
    ap = 0

  total_map += ap

total_map/20

0.975